# Master instruction —Feature Engineering-breed

Je werkt als senior research assistant voor een masterthesis in data analysis.
Voor meer informatie over de thesis / onderzoeksvoorstel / opzet: bekijk bron "Geannoteerd_onderzoeksvoorstel.md" en voor extra gelinkte literatuur bron; "External factors and SHAP in Urban Parking copy.pdf" (beide bestanden zijn te vinden in de map 'literatuur_en_info' (binnen dit project)) 

voor structuur en gewenste flow check projectalomvattede: "README.md"

Context:
- Check hieronder de vooropgestelde feature engineering notebookstructuur:
  1. fe_00_design_and_split_lock.ipynb
  2. fe_01_build_core_features_TSE.ipynb
  3. fe_02_build_forecast_lag_features_L.ipynb
  4. fe_03_finalize_feature_sets_and_export.ipynb
- Projectfase: Phase 3 — Feature Engineering
- Phase 2 (EDA) is afgerond, dit is DE basis voor interne consistentie en keuzes voor FE!!
  - Belangrijk is ook gelijkaardige stijl als phase 2 in phase 3 over te nemen, maar dan specifiek op feature engineering toegepast natuurlijk
- shortterm is de primaire modelbasis
  - (longterm blijft enkel robustness-context en mag niet het hoofddesign sturen)
- Er zijn twee modeltracks (meer uitleg is te vinden in: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/literatuur_en_info/Geannoteerd_onderzoeksvoorstel copy.md):
  1. forecast track: autoregressieve features toegestaan
  2. policy track: occupancy-lags NIET toegestaan
- De operationele split ligt vast:
  - train = 2020, 2023, 2024
  - holdout = 2025
  - 2019 uitgesloten wegens volledig missende weatherfeatures
- Alle transformaties moeten train-only gefit worden
- Alle tijdsfeatures moeten strikt causaal opgebouwd worden
- Model- en featurekeuze gebeurt later via rolling/blocked CV binnen train
- 2025 blijft volledig locked voor finale evaluatie

Doel van phase 3:
- een reproduceerbare, leakage-safe, immutable featureset bouwen
- consistente schema’s afleveren voor forecast en policy
- features expliciet structureren in blokken (cfr. onderzoeksvoorstel):
  - T = time structure
  - S = spatial identity
  - E = external factors
  - L = autoregressive structure (alleen forecast)
- alle featurekeuzes moeten inhoudelijk verdedigbaar zijn vanuit EDA en onderzoeksdoel

Belangrijke werkinstructies:
1. Werk notebook-native en reproduceerbaar.
2. Gebruik enkel train-only fitting voor alle fit-afhankelijke transformaties.
3. Splits train en holdout vroeg en hard; nooit lekken van 2025 naar train.
4. Maak feature engineering expliciet causaal:
   - geen toekomstinformatie
   - geen target-proxies
   - geen smoothing over toekomstige observaties
   - geen niet-causale aggregaties
5. Gebruik duidelijke helperfuncties en schema-validatie.
6. Werk primair op shortterm.
7. longterm mag enkel gebruikt worden voor beperkte robustness-context, niet als primaire pipelinebasis.
8. Maak alle featurebeslissingen expliciet en academisch verdedigbaar.
9. Sluit elk notebook af met:
   - "What was built"
   - "Leakage and validity checks"
   - "Implications for next notebook"
10. Indien iets ambigu is, kies de meest leakage-veilige optie en documenteer die.

Schrijf elke interpretatieve markdown-sectie alsof ze later kan worden herwerkt tot tekst voor de masterthesis.

Stijlregels:
- helder, academisch, voorzichtig
- geen losse bullet dump als lopende tekst beter is
- benoem richting, grootteorde, onzekerheid en beperking
- maak expliciet waarom het resultaat relevant is voor de volgende fase

Jouw taak is niet alleen code schrijven, maar een thesis-waardige feature engineering pipeline ontwerpen.

## Notebookspecifieke prompt
Maak notebook `fe_03_finalize_feature_sets_and_export.ipynb`.

Doel:
De definitieve feature sets finaliseren, valideren, documenteren en immutable exportbestanden opleveren voor Phase 4.

Context:
- TSE-featurelaag is gebouwd
- L-features zijn gebouwd voor forecast
- Er moeten twee definitieve feature sets ontstaan:
  - policy
  - forecast
- 2025 blijft locked holdout

Voer dit uit:
0. analyseer eerst fe_00, 01 en fe_02 zodat je intern consistent bent en blijft
1. Laad alle tussenoutputs uit fe_01 en fe_02.
2. Maak twee finale feature sets:
   - policy_final = T + S + E
   - forecast_final = T + S + E + L
3. Definieer expliciet:
   - target column
   - id columns
   - time index columns
   - model input columns
   - uitgesloten columns
4. Controleer streng:
   - identiek schema train vs holdout per track
   - dtype-consistentie
   - geen duplicate columns
   - geen constant/nearly-constant junk features
   - geen leakage-verdachte features
   - geen kwaliteitsflags als predictors
5. Maak feature manifests:
   - per track volledige featurelijst
   - bloklabels
   - allowed use
   - fit-dependency
   - causal status
6. Maak data dictionaries:
   - feature
   - betekenis
   - bron
   - transformatie
   - interpretatienoot
7. Maak exportbestanden:
   - policy_train.parquet
   - policy_holdout_2025.parquet
   - forecast_train.parquet
   - forecast_holdout_2025.parquet
   - feature_registry.csv
   - feature_manifest_policy.json
   - feature_manifest_forecast.json
   - schema snapshots
8. Voeg sanity-checks toe voor modelling readiness:
   - missings summary
   - per-track row counts
   - per-track feature counts
   - sample loss door lag validiteit
9. Schrijf een hand-off memo voor Phase 4:
   - welke feature sets bestaan?
   - welke hypothesen over FE zijn operationeel ingebouwd?
   - welke beperkingen moeten modelselectie respecteren?
   - welke ablaties zijn nu mogelijk?
10. Sluit af met:
   - "Final deliverables for Phase 4"
   - "Feature engineering choices that must not be violated later"
   - "Recommended modelling ablation map"

Belangrijk:
- Maak de output immutable en consistent.
- Dit notebook moet Phase 4 bijna plug-and-play maken.

## 0. Setup and finalization scope

This notebook finalizes Phase 3 into immutable, Phase-4-ready feature packages.

Contracts carried from previous notebooks:
- `fe_00`: split lock (`train = 2020, 2023, 2024`, `holdout = 2025`);
- `fe_01`: shared `T+S+E` layer;
- `fe_02`: forecast-only `L` layer with strict causal lag logic.

In [1]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_processed").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Project root not found")


def ensure_datetime(df: pd.DataFrame, col: str) -> None:
    df[col] = pd.to_datetime(df[col], errors="coerce")


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def schema_compare(train_df: pd.DataFrame, holdout_df: pd.DataFrame) -> pd.DataFrame:
    s_train = train_df.dtypes.astype(str).rename("dtype_train")
    s_hold = holdout_df.dtypes.astype(str).rename("dtype_holdout")
    out = pd.concat([s_train, s_hold], axis=1).reset_index().rename(columns={"index": "column"})
    out["dtype_match"] = out["dtype_train"].eq(out["dtype_holdout"])
    return out


def find_near_constant_features(df: pd.DataFrame, columns: list[str], threshold: float = 0.995) -> pd.DataFrame:
    rows = []
    for c in columns:
        vc = df[c].value_counts(dropna=False, normalize=True)
        top_freq = float(vc.iloc[0]) if len(vc) else np.nan
        n_unique = int(df[c].nunique(dropna=False))
        rows.append(
            {
                "feature_name": c,
                "n_unique": n_unique,
                "top_frequency": top_freq,
                "is_constant": n_unique <= 1,
                "is_near_constant": bool(top_freq >= threshold) if not np.isnan(top_freq) else False,
            }
        )
    return pd.DataFrame(rows)


def infer_transformation(feature_name: str, notes: str) -> str:
    fname = feature_name.lower()
    notes_l = str(notes).lower()
    if "lag" in fname:
        return "time-aware lag transform"
    if "roll" in fname:
        return "strict causal rolling window"
    if "_sin" in fname or "_cos" in fname:
        return "cyclical encoding"
    if "__" in fname and "int_" in fname:
        return "explicit interaction term"
    if "__" in fname:
        return "one-hot/bucket expansion"
    if "sq" in fname:
        return "non-linear polynomial transform"
    if "log" in fname:
        return "log transform"
    if "bin" in fname:
        return "binned categorical transform"
    if "threshold" in notes_l or "fit" in notes_l:
        return "threshold-based engineered transform"
    return "direct engineered feature"


PROJECT_ROOT = find_project_root()
PHASE3_SPLIT_DIR = PROJECT_ROOT / "data_processed" / "phase3_splits"
CORE_TSE_DIR = PROJECT_ROOT / "data_processed" / "phase3_features" / "core_tse"
FORECAST_L_DIR = PROJECT_ROOT / "data_processed" / "phase3_features" / "forecast_l"
OUT_DIR = PROJECT_ROOT / "data_processed" / "phase4_feature_sets"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "occupancy_rate"
ID_COLS = ["parking_id"]
TIME_COLS = ["rounded_hour", "year"]
SYSTEM_COLS = ["operational_split"]
BASE_COLS = ID_COLS + TIME_COLS + [TARGET_COL] + SYSTEM_COLS

print("Project root:", PROJECT_ROOT)
print("Final output dir:", OUT_DIR)

Project root: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2
Final output dir: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_processed/phase4_feature_sets


## 1. Analyze consistency with `fe_00`, `fe_01`, `fe_02`

This section explicitly checks that finalization starts from internally consistent upstream outputs.

In [2]:
split_summary_df = pd.read_csv(PHASE3_SPLIT_DIR / "shortterm_operational_split_summary.csv")
core_manifest_df = pd.read_csv(CORE_TSE_DIR / "tse_export_manifest.csv")
forecast_manifest_df = pd.read_csv(FORECAST_L_DIR / "forecast_l_export_manifest.csv")

consistency_rows = [
    {
        "source_notebook": "fe_00",
        "contract": "locked split exists",
        "status": "PASS" if set(split_summary_df["operational_split"]) == {"train", "holdout"} else "CHECK",
        "evidence": f"split rows={len(split_summary_df)}",
    },
    {
        "source_notebook": "fe_01",
        "contract": "TSE exports exist",
        "status": "PASS" if {"train_tse_parquet", "holdout_tse_parquet"}.issubset(set(core_manifest_df["artifact"])) else "CHECK",
        "evidence": f"manifest artifacts={len(core_manifest_df)}",
    },
    {
        "source_notebook": "fe_02",
        "contract": "forecast L exports exist",
        "status": "PASS" if {"forecast_train_parquet", "forecast_holdout_parquet"}.issubset(set(forecast_manifest_df["artifact"])) else "CHECK",
        "evidence": f"manifest artifacts={len(forecast_manifest_df)}",
    },
]

consistency_df = pd.DataFrame(consistency_rows)

print("Internal consistency checks")
display(consistency_df)

print("Split summary from fe_00")
display(split_summary_df)

Internal consistency checks


,source_notebook,contract,status,evidence
0,fe_00,locked split exists,PASS,split rows=2
1,fe_01,TSE exports exist,PASS,manifest artifacts=6
2,fe_02,forecast L exports exist,PASS,manifest artifacts=6


Split summary from fe_00


,operational_split,n_rows,n_parkings,date_min,date_max,mean_occupancy,pct_of_phase3_scope
0,holdout,87600,10,2025-01-01,2025-12-31 23:00:00,0.400808,40.287532
1,train,129837,10,2020-01-01,2024-12-31 23:00:00,0.361632,59.712468


## 2. Load intermediate outputs (`fe_01` and `fe_02`)

In [3]:
policy_train_raw = pd.read_parquet(CORE_TSE_DIR / "tse_core_train_2020_2023_2024.parquet")
policy_holdout_raw = pd.read_parquet(CORE_TSE_DIR / "tse_core_holdout_2025.parquet")
forecast_train_raw = pd.read_parquet(FORECAST_L_DIR / "forecast_tsel_train_2020_2023_2024.parquet")
forecast_holdout_raw = pd.read_parquet(FORECAST_L_DIR / "forecast_tsel_holdout_2025.parquet")

for df in [policy_train_raw, policy_holdout_raw, forecast_train_raw, forecast_holdout_raw]:
    ensure_datetime(df, "rounded_hour")

feature_set_separation = json.loads((FORECAST_L_DIR / "feature_set_separation.json").read_text())

tse_registry_df = pd.read_csv(CORE_TSE_DIR / "feature_registry_tse.csv")
l_registry_df = pd.read_csv(FORECAST_L_DIR / "feature_registry_l_forecast.csv")

load_overview_df = pd.DataFrame(
    [
        {"dataset": "policy_train_raw", "rows": len(policy_train_raw), "cols": policy_train_raw.shape[1]},
        {"dataset": "policy_holdout_raw", "rows": len(policy_holdout_raw), "cols": policy_holdout_raw.shape[1]},
        {"dataset": "forecast_train_raw", "rows": len(forecast_train_raw), "cols": forecast_train_raw.shape[1]},
        {"dataset": "forecast_holdout_raw", "rows": len(forecast_holdout_raw), "cols": forecast_holdout_raw.shape[1]},
    ]
)

display(load_overview_df)

,dataset,rows,cols
0,policy_train_raw,129837,92
1,policy_holdout_raw,87600,92
2,forecast_train_raw,129837,103
3,forecast_holdout_raw,87600,103


## 3. Build final feature sets

- `policy_final = T + S + E`
- `forecast_final = T + S + E + L`

In [4]:
policy_model_inputs = feature_set_separation["policy_features"]
forecast_model_inputs = feature_set_separation["forecast_features"]
selected_l_features = feature_set_separation["selected_l_features"]

# Columns present in forecast raw but excluded from model inputs (e.g., validity helper flags)
forecast_excluded_non_input = sorted(
    [
        c for c in forecast_train_raw.columns
        if c not in set(BASE_COLS + forecast_model_inputs)
    ]
)

policy_export_cols = BASE_COLS + policy_model_inputs
forecast_export_cols = BASE_COLS + forecast_model_inputs

policy_final_train = policy_train_raw[policy_export_cols].copy()
policy_final_holdout = policy_holdout_raw[policy_export_cols].copy()

forecast_final_train = forecast_train_raw[forecast_export_cols].copy()
forecast_final_holdout = forecast_holdout_raw[forecast_export_cols].copy()

final_shape_df = pd.DataFrame(
    [
        {"track": "policy", "split": "train", "rows": len(policy_final_train), "cols": policy_final_train.shape[1]},
        {"track": "policy", "split": "holdout", "rows": len(policy_final_holdout), "cols": policy_final_holdout.shape[1]},
        {"track": "forecast", "split": "train", "rows": len(forecast_final_train), "cols": forecast_final_train.shape[1]},
        {"track": "forecast", "split": "holdout", "rows": len(forecast_final_holdout), "cols": forecast_final_holdout.shape[1]},
    ]
)

print("Final set overview")
display(final_shape_df)

Final set overview


,track,split,rows,cols
0,policy,train,129837,92
1,policy,holdout,87600,92
2,forecast,train,129837,97
3,forecast,holdout,87600,97


## 4. Explicit column roles per track

Required explicit definitions:
- target column
- id columns
- time index columns
- model input columns
- excluded columns

In [5]:
column_roles_df = pd.DataFrame(
    [
        {
            "track": "policy",
            "target_column": TARGET_COL,
            "id_columns": "|".join(ID_COLS),
            "time_index_columns": "|".join(TIME_COLS),
            "n_model_input_columns": len(policy_model_inputs),
            "excluded_columns": "|".join(sorted(set(policy_train_raw.columns) - set(policy_export_cols))),
        },
        {
            "track": "forecast",
            "target_column": TARGET_COL,
            "id_columns": "|".join(ID_COLS),
            "time_index_columns": "|".join(TIME_COLS),
            "n_model_input_columns": len(forecast_model_inputs),
            "excluded_columns": "|".join(forecast_excluded_non_input),
        },
    ]
)

display(column_roles_df)

,track,target_column,id_columns,time_index_columns,n_model_input_columns,excluded_columns
0,policy,occupancy_rate,parking_id,rounded_hour|year,87,
1,forecast,occupancy_rate,parking_id,rounded_hour|year,92,l_valid_all_core|l_valid_core_plus_roll24|l_va...


## 5. Strict validation checks

Checks required by design:
- train vs holdout schema equivalence per track
- dtype consistency
- no duplicate columns
- no constant / near-constant junk
- no leakage-suspect features
- no quality flags as predictors

In [6]:
policy_schema_df = schema_compare(policy_final_train, policy_final_holdout)
forecast_schema_df = schema_compare(forecast_final_train, forecast_final_holdout)

policy_dup_cols = [c for c in policy_final_train.columns if list(policy_final_train.columns).count(c) > 1]
forecast_dup_cols = [c for c in forecast_final_train.columns if list(forecast_final_train.columns).count(c) > 1]

policy_const_df = find_near_constant_features(policy_final_train, policy_model_inputs, threshold=0.995)
forecast_const_df = find_near_constant_features(forecast_final_train, forecast_model_inputs, threshold=0.995)

policy_bad_const = policy_const_df.loc[policy_const_df["is_constant"] | policy_const_df["is_near_constant"]]
forecast_bad_const = forecast_const_df.loc[forecast_const_df["is_constant"] | forecast_const_df["is_near_constant"]]

forbidden_name_tokens = [
    "quality", "blackout", "coverage", "partial", "event_names_combined", "number_of_occupied_spaces", "low_data_coverage", "system_blackout"
]

policy_leak_hits = sorted({c for c in policy_model_inputs for tok in forbidden_name_tokens if tok in c.lower()})
forecast_leak_hits = sorted({c for c in forecast_model_inputs for tok in forbidden_name_tokens if tok in c.lower()})

# Combine registry for source-level checks
l_selected_registry_df = l_registry_df.loc[l_registry_df["feature_name"].isin(selected_l_features)].copy()
combined_registry_df = pd.concat([tse_registry_df, l_selected_registry_df], axis=0, ignore_index=True)
combined_registry_df = combined_registry_df.drop_duplicates(subset=["feature_name"]).reset_index(drop=True)

source_joined = combined_registry_df.set_index("feature_name")["source_columns"].astype(str).to_dict()

forbidden_source_tokens = [
    "event_names_combined", "number_of_occupied_spaces", "low_data_coverage", "system_blackout", "partial_year", "flag_occ_inconsistent"
]

def source_hits(feature_names: list[str]) -> list[str]:
    hits = []
    for f in feature_names:
        src = source_joined.get(f, "")
        if any(tok in src for tok in forbidden_source_tokens):
            hits.append(f)
    return sorted(set(hits))

policy_source_leak_hits = source_hits(policy_model_inputs)
forecast_source_leak_hits = source_hits(forecast_model_inputs)

validation_df = pd.DataFrame(
    [
        {
            "check": "policy schema train vs holdout",
            "status": "PASS" if policy_schema_df["dtype_match"].all() else "FAIL",
            "detail": int((~policy_schema_df["dtype_match"]).sum()),
        },
        {
            "check": "forecast schema train vs holdout",
            "status": "PASS" if forecast_schema_df["dtype_match"].all() else "FAIL",
            "detail": int((~forecast_schema_df["dtype_match"]).sum()),
        },
        {
            "check": "duplicate columns in policy",
            "status": "PASS" if len(policy_dup_cols) == 0 else "FAIL",
            "detail": len(policy_dup_cols),
        },
        {
            "check": "duplicate columns in forecast",
            "status": "PASS" if len(forecast_dup_cols) == 0 else "FAIL",
            "detail": len(forecast_dup_cols),
        },
        {
            "check": "constant/near-constant junk in policy",
            "status": "PASS" if len(policy_bad_const) == 0 else "FAIL",
            "detail": len(policy_bad_const),
        },
        {
            "check": "constant/near-constant junk in forecast",
            "status": "PASS" if len(forecast_bad_const) == 0 else "FAIL",
            "detail": len(forecast_bad_const),
        },
        {
            "check": "name-level leakage suspicious policy predictors",
            "status": "PASS" if len(policy_leak_hits) == 0 else "FAIL",
            "detail": len(policy_leak_hits),
        },
        {
            "check": "name-level leakage suspicious forecast predictors",
            "status": "PASS" if len(forecast_leak_hits) == 0 else "FAIL",
            "detail": len(forecast_leak_hits),
        },
        {
            "check": "source-level leakage suspicious policy predictors",
            "status": "PASS" if len(policy_source_leak_hits) == 0 else "FAIL",
            "detail": len(policy_source_leak_hits),
        },
        {
            "check": "source-level leakage suspicious forecast predictors",
            "status": "PASS" if len(forecast_source_leak_hits) == 0 else "FAIL",
            "detail": len(forecast_source_leak_hits),
        },
        {
            "check": "policy track contains L-feature",
            "status": "PASS" if not any(c.startswith("l_") for c in policy_model_inputs) else "FAIL",
            "detail": int(any(c.startswith("l_") for c in policy_model_inputs)),
        },
    ]
)

print("Validation summary")
display(validation_df)

print("Policy schema check")
display(policy_schema_df)

print("Forecast schema check")
display(forecast_schema_df)

if len(policy_bad_const) > 0:
    print("Policy constant/near-constant features")
    display(policy_bad_const)

if len(forecast_bad_const) > 0:
    print("Forecast constant/near-constant features")
    display(forecast_bad_const)

assert validation_df["status"].eq("PASS").all(), "One or more strict finalization checks failed"

Validation summary


,check,status,detail
0,policy schema train vs holdout,PASS,0
1,forecast schema train vs holdout,PASS,0
2,duplicate columns in policy,PASS,0
3,duplicate columns in forecast,PASS,0
4,constant/near-constant junk in policy,PASS,0
5,constant/near-constant junk in forecast,PASS,0
6,name-level leakage suspicious policy predictors,PASS,0
7,name-level leakage suspicious forecast predictors,PASS,0
8,source-level leakage suspicious policy predictors,PASS,0
9,source-level leakage suspicious forecast predi...,PASS,0


Policy schema check


,column,dtype_train,dtype_holdout,dtype_match
0,parking_id,str,str,True
1,rounded_hour,datetime64[ns],datetime64[ns],True
2,year,Int64,Int64,True
3,occupancy_rate,float64,float64,True
4,operational_split,str,str,True
5,e_event_proximity_clip,int64,int64,True
6,e_event_scale_is_large,int8,int8,True
7,e_event_scale_max,int64,int64,True
8,e_event_window_post_24h,int64,int64,True
9,e_event_window_pre_24h,int64,int64,True


Forecast schema check


,column,dtype_train,dtype_holdout,dtype_match
0,parking_id,str,str,True
1,rounded_hour,datetime64[ns],datetime64[ns],True
2,year,Int64,Int64,True
3,occupancy_rate,float64,float64,True
4,operational_split,str,str,True
5,e_event_proximity_clip,int64,int64,True
6,e_event_scale_is_large,int8,int8,True
7,e_event_scale_max,int64,int64,True
8,e_event_window_post_24h,int64,int64,True
9,e_event_window_pre_24h,int64,int64,True


## 6. Final feature registry, manifests, and data dictionaries

Required metadata outputs:
- unified `feature_registry.csv`
- `feature_manifest_policy.json`
- `feature_manifest_forecast.json`
- per-track data dictionaries

In [7]:
# Build unified final registry
final_registry_df = combined_registry_df.copy()
final_registry_df["in_policy_final"] = final_registry_df["feature_name"].isin(policy_model_inputs)
final_registry_df["in_forecast_final"] = final_registry_df["feature_name"].isin(forecast_model_inputs)
final_registry_df["selected_for_final_export"] = final_registry_df["in_policy_final"] | final_registry_df["in_forecast_final"]
final_registry_df = final_registry_df.loc[final_registry_df["selected_for_final_export"]].copy()

final_registry_df = final_registry_df.sort_values(["block", "feature_name"]).reset_index(drop=True)

# Dictionaries
for_dict = final_registry_df.copy()
for_dict["transformatie"] = for_dict.apply(lambda r: infer_transformation(str(r["feature_name"]), str(r.get("notes", ""))), axis=1)
for_dict = for_dict.rename(
    columns={
        "feature_name": "feature",
        "expected_role": "betekenis",
        "source_columns": "bron",
        "notes": "interpretatienoot",
    }
)

policy_dictionary_df = for_dict.loc[for_dict["feature"].isin(policy_model_inputs), ["feature", "betekenis", "bron", "transformatie", "interpretatienoot"]].copy()
forecast_dictionary_df = for_dict.loc[for_dict["feature"].isin(forecast_model_inputs), ["feature", "betekenis", "bron", "transformatie", "interpretatienoot"]].copy()

# Feature manifests with full metadata per track
registry_idx = final_registry_df.set_index("feature_name")

def build_track_manifest(track_name: str, input_cols: list[str], excluded_cols: list[str]) -> dict:
    feature_items = []
    for c in input_cols:
        row = registry_idx.loc[c]
        feature_items.append(
            {
                "feature_name": c,
                "block": row["block"],
                "allowed_use": row["track_allowed"],
                "requires_fit": bool(row["requires_fit"]),
                "train_only_fit": bool(row["train_only_fit"]),
                "causal": bool(row["causal"]),
                "source_columns": row["source_columns"],
                "expected_role": row["expected_role"],
                "notes": row["notes"],
            }
        )

    return {
        "track": track_name,
        "target_column": TARGET_COL,
        "id_columns": ID_COLS,
        "time_index_columns": TIME_COLS,
        "system_columns": SYSTEM_COLS,
        "model_input_columns": input_cols,
        "excluded_columns": excluded_cols,
        "n_model_inputs": len(input_cols),
        "features": feature_items,
    }

policy_manifest = build_track_manifest(
    track_name="policy",
    input_cols=policy_model_inputs,
    excluded_cols=sorted(set(policy_train_raw.columns) - set(policy_export_cols)),
)

forecast_manifest = build_track_manifest(
    track_name="forecast",
    input_cols=forecast_model_inputs,
    excluded_cols=forecast_excluded_non_input,
)

manifest_overview_df = pd.DataFrame(
    [
        {"track": "policy", "n_model_inputs": len(policy_manifest["model_input_columns"]), "n_excluded_columns": len(policy_manifest["excluded_columns"])},
        {"track": "forecast", "n_model_inputs": len(forecast_manifest["model_input_columns"]), "n_excluded_columns": len(forecast_manifest["excluded_columns"])},
    ]
)

print("Manifest overview")
display(manifest_overview_df)

print("Final registry block summary")
display(final_registry_df.groupby(["block"], as_index=False).size().rename(columns={"size": "n_features"}))

Manifest overview


,track,n_model_inputs,n_excluded_columns
0,policy,87,0
1,forecast,92,6


Final registry block summary


,block,n_features
0,E,49
1,L,5
2,S,17
3,T,21


## 7. Export immutable Phase 4 deliverables

Requested exports:
- `policy_train.parquet`
- `policy_holdout_2025.parquet`
- `forecast_train.parquet`
- `forecast_holdout_2025.parquet`
- `feature_registry.csv`
- `feature_manifest_policy.json`
- `feature_manifest_forecast.json`
- schema snapshots

In [8]:
# Deterministic row order for immutability
sort_cols = ["parking_id", "rounded_hour"]
policy_final_train = policy_final_train.sort_values(sort_cols).reset_index(drop=True)
policy_final_holdout = policy_final_holdout.sort_values(sort_cols).reset_index(drop=True)
forecast_final_train = forecast_final_train.sort_values(sort_cols).reset_index(drop=True)
forecast_final_holdout = forecast_final_holdout.sort_values(sort_cols).reset_index(drop=True)

policy_train_path = OUT_DIR / "policy_train.parquet"
policy_holdout_path = OUT_DIR / "policy_holdout_2025.parquet"
forecast_train_path = OUT_DIR / "forecast_train.parquet"
forecast_holdout_path = OUT_DIR / "forecast_holdout_2025.parquet"

feature_registry_path = OUT_DIR / "feature_registry.csv"
manifest_policy_path = OUT_DIR / "feature_manifest_policy.json"
manifest_forecast_path = OUT_DIR / "feature_manifest_forecast.json"

schema_policy_path = OUT_DIR / "schema_policy_snapshot.json"
schema_forecast_path = OUT_DIR / "schema_forecast_snapshot.json"

dict_policy_path = OUT_DIR / "data_dictionary_policy.csv"
dict_forecast_path = OUT_DIR / "data_dictionary_forecast.csv"

policy_final_train.to_parquet(policy_train_path, index=False)
policy_final_holdout.to_parquet(policy_holdout_path, index=False)
forecast_final_train.to_parquet(forecast_train_path, index=False)
forecast_final_holdout.to_parquet(forecast_holdout_path, index=False)

final_registry_df.to_csv(feature_registry_path, index=False)
manifest_policy_path.write_text(json.dumps(policy_manifest, indent=2))
manifest_forecast_path.write_text(json.dumps(forecast_manifest, indent=2))

schema_policy_payload = {
    "columns": policy_final_train.columns.tolist(),
    "dtypes": {c: str(t) for c, t in policy_final_train.dtypes.items()},
}
schema_forecast_payload = {
    "columns": forecast_final_train.columns.tolist(),
    "dtypes": {c: str(t) for c, t in forecast_final_train.dtypes.items()},
}

schema_policy_path.write_text(json.dumps(schema_policy_payload, indent=2))
schema_forecast_path.write_text(json.dumps(schema_forecast_payload, indent=2))

policy_dictionary_df.to_csv(dict_policy_path, index=False)
forecast_dictionary_df.to_csv(dict_forecast_path, index=False)

# Immutable checksum manifest
artifact_paths = [
    policy_train_path,
    policy_holdout_path,
    forecast_train_path,
    forecast_holdout_path,
    feature_registry_path,
    manifest_policy_path,
    manifest_forecast_path,
    schema_policy_path,
    schema_forecast_path,
    dict_policy_path,
    dict_forecast_path,
]

artifact_rows = []
for p in artifact_paths:
    artifact_rows.append(
        {
            "artifact": p.name,
            "path": str(p),
            "bytes": p.stat().st_size,
            "sha256": file_sha256(p),
        }
    )

immutable_manifest_df = pd.DataFrame(artifact_rows)
immutable_manifest_path = OUT_DIR / "immutable_export_manifest.csv"
immutable_manifest_df.to_csv(immutable_manifest_path, index=False)

export_overview_df = pd.DataFrame(
    [
        {"track": "policy", "split": "train", "rows": len(policy_final_train), "n_model_inputs": len(policy_model_inputs)},
        {"track": "policy", "split": "holdout", "rows": len(policy_final_holdout), "n_model_inputs": len(policy_model_inputs)},
        {"track": "forecast", "split": "train", "rows": len(forecast_final_train), "n_model_inputs": len(forecast_model_inputs)},
        {"track": "forecast", "split": "holdout", "rows": len(forecast_final_holdout), "n_model_inputs": len(forecast_model_inputs)},
    ]
)

print("Export overview")
display(export_overview_df)

print("Immutable artifact manifest")
display(immutable_manifest_df)

Export overview


,track,split,rows,n_model_inputs
0,policy,train,129837,87
1,policy,holdout,87600,87
2,forecast,train,129837,92
3,forecast,holdout,87600,92


Immutable artifact manifest


,artifact,path,bytes,sha256
0,policy_train.parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,2447741,8c8ead6b8ed69846c1de0d5c876db60e20df4030380ee7...
1,policy_holdout_2025.parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,1217876,490eeeb6fd8a3b92c0756b5ee12c8cfde8a443157eaa1e...
2,forecast_train.parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,3387576,c7be991a115a1f2f63b25dcf1a048c1ba648993f141f81...
3,forecast_holdout_2025.parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,1886000,493d830117ad45281b1a47679c0a8c0ec2204e051eeb4f...
4,feature_registry.csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,20234,7db132f9d1ef1e49f3985b1704e5bdb67fa663813d3317...
5,feature_manifest_policy.json,/Users/emilevandevoorde/Documents/mp_mechelen_...,39318,c3962e21e6da2401555ed7dce135e39e2b885d50841bb2...
6,feature_manifest_forecast.json,/Users/emilevandevoorde/Documents/mp_mechelen_...,41368,b693120448ea6fb149eb662173d92063e64f083f6b16c4...
7,schema_policy_snapshot.json,/Users/emilevandevoorde/Documents/mp_mechelen_...,6033,f75e4026b2337f9eaa5028af3591eb5f9ab0ee5816986a...
8,schema_forecast_snapshot.json,/Users/emilevandevoorde/Documents/mp_mechelen_...,6338,bf88362229385640e81f0693d5b70959c8dba9ecbd942f...
9,data_dictionary_policy.csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,17301,a5f5bfbb79001a1d6adf89cfbfbd527752ab99188a0017...


## 8. Modelling-readiness sanity checks

Includes:
- missing summary per track
- row counts and feature counts
- sample loss due lag validity

In [9]:
def missing_summary(df: pd.DataFrame, feature_cols: list[str], split_label: str, track: str) -> pd.DataFrame:
    m = df[feature_cols].isna().mean().sort_values(ascending=False)
    return pd.DataFrame(
        {
            "track": track,
            "split": split_label,
            "feature_name": m.index,
            "missing_rate": m.values,
        }
    )

missing_policy_train_df = missing_summary(policy_final_train, policy_model_inputs, "train", "policy")
missing_policy_holdout_df = missing_summary(policy_final_holdout, policy_model_inputs, "holdout", "policy")
missing_forecast_train_df = missing_summary(forecast_final_train, forecast_model_inputs, "train", "forecast")
missing_forecast_holdout_df = missing_summary(forecast_final_holdout, forecast_model_inputs, "holdout", "forecast")

row_feature_counts_df = pd.DataFrame(
    [
        {"track": "policy", "split": "train", "rows": len(policy_final_train), "feature_count": len(policy_model_inputs)},
        {"track": "policy", "split": "holdout", "rows": len(policy_final_holdout), "feature_count": len(policy_model_inputs)},
        {"track": "forecast", "split": "train", "rows": len(forecast_final_train), "feature_count": len(forecast_model_inputs)},
        {"track": "forecast", "split": "holdout", "rows": len(forecast_final_holdout), "feature_count": len(forecast_model_inputs)},
    ]
)

# Sample loss due lag validity (based on core lag availability)
core_lag_cols = ["l_occ_lag_1h", "l_occ_lag_24h", "l_occ_lag_168h"]
forecast_train_core_valid = forecast_final_train[core_lag_cols].notna().all(axis=1)
forecast_holdout_core_valid = forecast_final_holdout[core_lag_cols].notna().all(axis=1)

sample_loss_lag_df = pd.DataFrame(
    [
        {
            "split": "train",
            "rows_total": len(forecast_final_train),
            "rows_valid_all_core_lags": int(forecast_train_core_valid.sum()),
            "rows_dropped_if_strict_core_lag_filter": int((~forecast_train_core_valid).sum()),
            "pct_dropped_if_strict_core_lag_filter": float((~forecast_train_core_valid).mean() * 100),
        },
        {
            "split": "holdout",
            "rows_total": len(forecast_final_holdout),
            "rows_valid_all_core_lags": int(forecast_holdout_core_valid.sum()),
            "rows_dropped_if_strict_core_lag_filter": int((~forecast_holdout_core_valid).sum()),
            "pct_dropped_if_strict_core_lag_filter": float((~forecast_holdout_core_valid).mean() * 100),
        },
    ]
)

print("Row/feature counts")
display(row_feature_counts_df)

print("Sample loss due lag validity")
display(sample_loss_lag_df.round(4))

print("Missing summary - policy train (top 15)")
display(missing_policy_train_df.head(15))

print("Missing summary - forecast train (top 15)")
display(missing_forecast_train_df.head(15))

Row/feature counts


,track,split,rows,feature_count
0,policy,train,129837,87
1,policy,holdout,87600,87
2,forecast,train,129837,92
3,forecast,holdout,87600,92


Sample loss due lag validity


,split,rows_total,rows_valid_all_core_lags,rows_dropped_if_strict_core_lag_filter,pct_dropped_if_strict_core_lag_filter
0,train,129837,108519,21318,16.419
1,holdout,87600,87600,0,0.000


Missing summary - policy train (top 15)


,track,split,feature_name,missing_rate
0,policy,train,e_event_proximity_clip,0.0
1,policy,train,s_parking_id__p_hoogstraat,0.0
2,policy,train,s_tier_is_centrum,0.0
3,policy,train,s_parking_id__p_veemarkt,0.0
4,policy,train,s_parking_id__p_tinel,0.0
5,policy,train,s_parking_id__p_maarten,0.0
6,policy,train,s_parking_id__p_lamot,0.0
7,policy,train,s_parking_id__p_komet,0.0
8,policy,train,s_parking_id__p_keerdok,0.0
9,policy,train,s_parking_id__p_kathedraal,0.0


Missing summary - forecast train (top 15)


,track,split,feature_name,missing_rate
0,forecast,train,l_occ_lag_delta_24h_168h,0.135955
1,forecast,train,l_occ_lag_delta_1h_24h,0.112587
2,forecast,train,l_occ_lag_168h,0.091576
3,forecast,train,l_occ_lag_24h,0.074093
4,forecast,train,l_occ_lag_1h,0.057719
5,forecast,train,s_parking_id__p_keerdok,0.000000
6,forecast,train,s_tier_is_vesten,0.000000
7,forecast,train,s_tier_is_rand,0.000000
8,forecast,train,s_tier_is_centrum,0.000000
9,forecast,train,s_parking_id__p_veemarkt,0.000000


## 9. Phase 4 hand-off memo

This memo is written as a modelling hand-off contract.

In [10]:
handoff_rows = [
    {
        "question": "Welke feature sets bestaan?",
        "answer": "Policy set (T+S+E) en Forecast set (T+S+E+L), elk met train en locked holdout 2025 exports.",
    },
    {
        "question": "Welke hypothesen over FE zijn operationeel ingebouwd?",
        "answer": "Cyclische tijdsstructuur, spatial heterogeniteit, niet-lineaire externe drivers, en forecast-only autoregressieve geheugenstructuur (lags).",
    },
    {
        "question": "Welke beperkingen moet modelselectie respecteren?",
        "answer": "Geen leakage over holdout, policy mag geen L-features gebruiken, train-only fitregels uit fe_01 blijven bindend.",
    },
    {
        "question": "Welke ablaties zijn nu mogelijk?",
        "answer": "Policy: TS vs TSE; Forecast: TSE vs TSEL(core lags) en optioneel TSEL+strict rolling als robustness variant.",
    },
]

handoff_memo_df = pd.DataFrame(handoff_rows)
display(handoff_memo_df)

,question,answer
0,Welke feature sets bestaan?,"Policy set (T+S+E) en Forecast set (T+S+E+L), ..."
1,Welke hypothesen over FE zijn operationeel ing...,"Cyclische tijdsstructuur, spatial heterogenite..."
2,Welke beperkingen moet modelselectie respecteren?,"Geen leakage over holdout, policy mag geen L-f..."
3,Welke ablaties zijn nu mogelijk?,Policy: TS vs TSE; Forecast: TSE vs TSEL(core ...


## Final deliverables for Phase 4

1. Immutable policy feature exports:
   - `policy_train.parquet`
   - `policy_holdout_2025.parquet`
2. Immutable forecast feature exports:
   - `forecast_train.parquet`
   - `forecast_holdout_2025.parquet`
3. Metadata and governance artifacts:
   - `feature_registry.csv`
   - `feature_manifest_policy.json`
   - `feature_manifest_forecast.json`
   - `schema_policy_snapshot.json`
   - `schema_forecast_snapshot.json`
   - `immutable_export_manifest.csv`
4. Documentation artifacts:
   - `data_dictionary_policy.csv`
   - `data_dictionary_forecast.csv`

## Feature engineering choices that must not be violated later

1. `2025` remains a locked holdout for final evaluation and must not be used for fitting or tuning.
2. Policy models must use only `T+S+E`; forecast-only `L` features are prohibited in policy track.
3. Lag features must remain exact time-aware and parking-aware; no forward fill, no hidden imputation.
4. Feature schemas and dtypes from exported snapshots are contractual for Phase 4 pipelines.
5. Quality flags and target proxies remain excluded as predictors.

## Recommended modelling ablation map

1. Policy track:
   - `TS`
   - `TSE` (final policy set)
2. Forecast track:
   - `TSE` baseline without `L`
   - `TSEL-core` (selected lag set, default forecast)
   - `TSEL-core+strict-rolling` (optional robustness variant with explicit sample-loss reporting)
3. Cross-track interpretation:
   - compare policy vs forecast to separate scenario-sensitive signals from inertial predictive gain.

## What was built

The notebook finalized both feature tracks, enforced strict schema and leakage checks, generated manifests/dictionaries, and exported immutable Phase-4-ready artifacts with checksums.

## Leakage and validity checks

Schema equivalence, duplicate-column checks, constant-feature checks, leakage token scans, and lag-validity impact summaries were executed before export.

## Implications for next notebook

Phase 4 can now consume the finalized exports directly for modelling, ablation experiments, and locked 2025 evaluation without redefining FE contracts.